# df_alter Code

In [ ]:
import networkx as nx 
import pandas as pd 
from pathlib import Path

# setting the directory path
data_dir = Path('/Users/howardleong/Desktop/academic/Fundamentals of Social Data Science in Python/Week 2/Network Canvas Export Tables')  

# function to load csv file
def load_csv_files(pattern):
    dfs = []
    for file in data_dir.glob(pattern):
        df = pd.read_csv(file)
        df['source_file'] = file.name
        dfs.append(df)
    
    if dfs:
        return pd.concat(dfs, ignore_index=True)
        
    return None

# Load different types of files
attr_df = load_csv_files("*_attributeList_concept.csv")
ego_df = load_csv_files("*_ego.csv")
edge_df = load_csv_files("*_edgeList_relate.csv")

# get the SessionID from attriubute file source_file column
attr_df['SessionID'] = attr_df['source_file'].str.extract(r'Anonymous Participant_([\w]+)_attributeList_concept\.csv')
ego_df['SessionID'] = ego_df['source_file'].str.extract(r'Anonymous Participant_([\w]+)_ego\.csv')

# merge attribute file with ego file based on SessionID
df_alter = pd.merge(attr_df, ego_df, on='SessionID', suffixes=('_attr', '_ego'))
df_alter["name"].value_counts()

# build a dictionary {"sessionID": file_path}
edge_list = []
for i in data_dir.glob("*ego.csv"):
    x = i.name.split("_")[:-1]
    x.append("edgeList_relate.csv")
    edge_path = Path(data_dir / "_".join(x))
    key = i.name.split("_")[1]
    
    if edge_path.exists():
        edge_list.append(
            {"key": key,
             "path": edge_path
             })
    else:
        raise Exception("Warning: The edge path file does not exist.") 


# read the edge files and build graphs
edgelist_dfs = []
for edge_file in edge_list:
    edge_df = pd.read_csv(edge_file["path"])
    edges = zip(edge_df["networkCanvasSourceUUID"].to_list(),edge_df["networkCanvasTargetUUID"].to_list())
    edges = list(edges)
    G = nx.Graph(edges) # put it into a networkx graph

    # Find the connected components, sort them by size (largest first)
    ccs = sorted(list(nx.connected_components(G)), key=len, reverse=True)

    # Get the largest component (or components, in case of a tie)
    giant_component = ccs[0]
    if len(ccs) > 1:
        for i in ccs[1:]:
            if len(i) == len(ccs[0]):
                giant_component.extend(i)

    # Create a dataframe for the nodes in the giant component
    df_temp = pd.DataFrame(giant_component, columns=["networkCanvasUUID"])
    df_temp["networkCanvasSessionID"] = edge_file["key"]
    df_temp["in_gcc"] = 1 # tag as in gcc 
    edgelist_dfs.append(df_temp)

# Combine all the dataframes and merge with df_alters
df_edgelist_gcc = pd.concat(edgelist_dfs) 
df_alter = df_alter.merge(df_edgelist_gcc, on=["networkCanvasSessionID","networkCanvasUUID"], how="left")

# Where we didn't get a value from the merge, that node was not in the giant component for that ego
df_alter.fillna({"in_gcc":0}, inplace = True)
display(df_alter)

# Relative Distance

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean
from collections import defaultdict

def relative_distance(df_alter):
    
    # Get unique alter names
    unique_alters = df_alter['name'].unique()

    # Dictionary to store distances for each pair across sessions
    pair_distances = defaultdict(list)

    # Calculate distances for each session
    for session_id in df_alter['SessionID'].unique():
        df_temp = df_alter[df_alter['SessionID'] == session_id]
        
        for i in df_temp['name'].unique():
            for j in df_temp['name'].unique():
                if i != j:
                    vec_i = df_temp[df_temp['name'] == i][['user_layout_x', 'user_layout_y']].values.flatten()
                    vec_j = df_temp[df_temp['name'] == j][['user_layout_x', 'user_layout_y']].values.flatten()
                    
                    # Only calculate if both vectors exist and are not empty
                    if len(vec_i) > 0 and len(vec_j) > 0:
                        dist = euclidean(vec_i, vec_j)
                        pair_distances[(i, j)].append(dist)

    # Create dataframe and populate with averaged distances
    df_relative = pd.DataFrame(index=unique_alters, columns=unique_alters)

    for (i, j), distances in pair_distances.items():
        df_relative.loc[i, j] = np.mean(distances)

    # Optional: fill diagonal with 0 (distance from alter to itself)
    np.fill_diagonal(df_relative.values, 0)

    display(df_relative)

    return df_relative

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_relative_distance_heatmap(df_relative):
    # Create a mask for values >= 0.2
    mask = df_relative.astype(float) >= 0.2

    # Create heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(df_relative.astype(float), 
                annot=False,  # Don't show values
                cmap='YlOrRd_r',
                mask=mask,  # Hide values >= 0.2
                cbar_kws={'label': 'Euclidean Distance'},
                linewidths=0.5,
                square=True,
                vmin=0,
                vmax=0.2)  # Set color scale to 0-0.2 range

    plt.title('Alter Pairs with Distance < 0.2', fontsize=16, pad=20)
    plt.xlabel('Alter Name', fontsize=12)
    plt.ylabel('Alter Name', fontsize=12)
    plt.tight_layout()
    plt.show()

df_relative = relative_distance(df_alter)
plot_relative_distance_heatmap(df_relative)

# Analysis

In [ ]:
# filter out the index where the relative distance is less that 0.05
for_ai = df_relative.index[df_relative["AI"] > 0.3].tolist()
for_ai

# filter for SessionID where for_ai nodes are present
sessions_far_ai = (
    df_alter.loc[df_alter['name'].isin(for_ai), 'SessionID']
    .dropna()
    .unique()
    .tolist()
)
all_sessions = df_alter['SessionID'].dropna().unique().tolist()
sessions_close_ai = [s for s in all_sessions if s not in sessions_far_ai]

degree_close_ai = [] 
degree_far_ai = [] 

for session in all_sessions: 

    df_temp = df_alter[df_alter['SessionID'] == session]
    edge_file_path = data_dir / f'Anonymous Participant_{session}_edgeList_relate.csv'
    edge_df = pd.read_csv(edge_file_path)
    edges_named = (
        edge_df
        .merge(df_alter[['networkCanvasUUID', 'name']], how='left',
            left_on='networkCanvasSourceUUID', right_on='networkCanvasUUID')
        .rename(columns={'name': 'source_name'})
        .merge(df_alter[['networkCanvasUUID', 'name']], how='left',
            left_on='networkCanvasTargetUUID', right_on='networkCanvasUUID')
        .rename(columns={'name': 'target_name'})
    )
    G = nx.from_pandas_edgelist(
        edges_named,
        source='source_name',
        target='target_name'
    )

    if session in sessions_close_ai: 
        degree_close_ai.append(sum(dict(G.degree()).values()) / G.number_of_nodes())
    elif session in sessions_far_ai: 
        degree_far_ai.append(sum(dict(G.degree()).values()) / G.number_of_nodes())

# plot bar chart with error bars for density_close_ai and density_far_ai
means = [np.mean(degree_close_ai), np.mean(degree_far_ai)]
errors = [np.std(degree_close_ai), np.std(degree_far_ai)]
labels = ['Networks without Fuzzy Concepts', 'Networks with Fuzzy Concepts']

plt.figure(figsize=(6, 5))
plt.bar(labels, means, yerr=errors, capsize=6, color=['#4c72b0', '#dd8452'], alpha=0.8)
plt.ylabel("Network Average Degree")
plt.title("Comparison of Average Degree")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()